# Train TrafficCAM loop (Free T4 — the only GPU step)

Fine-tunes `yolov8n` on converted TrafficCAM (+ ITD later, same converter interface) and packages a registry candidate.
The finale stays canonical unless the candidate wins on measurement (gates in Cell 5).

| | |
|---|---|---
| GPU | **Required here.** Runtime → T4. Everything else in this notebook is CPU. |
| Prereq | Run `eval_trafficcam_drive.ipynb` Cells 0–3 first, **or** re-run Cells 0–2 here (idempotent). |
| Upload | `best_f007.pt` (loop-8 weights) to `/content` when asked — same Drive contract as the BMD finale. |
| Hand-back | `trafficcam_candidate_<date>.zip` (fp32 + int8 + metadata.json + val report) → `notebooks/training_output_zips/` |

Resume-safe: re-running prep cells skips existing outputs.

In [ ]:
# Cell 0 — GPU check + setup (fail fast without T4)
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU, then re-run'
print('cuda:', torch.cuda.get_device_name(0))
!pip install -q ultralytics onnxruntime opencv-python-headless gdown
!apt-get install -y -qq git-lfs > /dev/null 2>&1; git lfs install --skip-repo > /dev/null 2>&1
import os, shutil, subprocess, sys
REPO = '/content/SGP-IV'
BRANCH = 'feat/policy-ports-screenshot-green'
def _git(*a):
    return subprocess.run(['git', '-C', REPO, *a], capture_output=True, text=True)
_cur = _git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip() \
    if os.path.isdir(REPO + '/.git') else ''
if _cur != BRANCH:  # stale main-clone or foreign checkout: replace wholesale
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/Bobbymkr/SGP-IV.git', REPO], check=True)
print('repo:', _git('rev-parse', '--abbrev-ref', 'HEAD').stdout.strip(),
      _git('rev-parse', '--short', 'HEAD').stdout.strip())
print('setup ok')
# fingerprint: paste this SHA with any bug report — proves which code ran
!cd /content/SGP-IV && git rev-parse --short HEAD && git log --oneline -1


In [ ]:
# Cell 1 — data: download + convert (idempotent; skip if /content/tcam exists)
import os
if not os.path.exists('/content/tcam/data.yaml'):
    # the shell cwd on each line, so relative cd/gdown/unzip silently misplaces files)
    !mkdir -p /content/trafficcam
    !test -f /content/trafficcam/Fully_annotate.zip || gdown https://drive.google.com/uc?id=1h4oUqECDF05vSYMkYgz0aUc_RRzHlh3e -O /content/trafficcam/Fully_annotate.zip
    !test -f /content/trafficcam/splits.zip || gdown https://drive.google.com/uc?id=1GcQ3J56kU-6TwRqRjzvI28-OF6QUdHeR -O /content/trafficcam/splits.zip
    # rescue: an older revision of this cell dropped zips in /content — reuse them
    !test -f /content/Fully_annotate.zip && cp -n /content/Fully_annotate.zip /content/trafficcam/ ; true
    !test -f /content/splits.zip && cp -n /content/splits.zip /content/trafficcam/ ; true
    !unzip -q -o /content/trafficcam/Fully_annotate.zip -d /content/trafficcam
    !unzip -q -o /content/trafficcam/splits.zip -d /content/trafficcam
    import glob
    _n = len(glob.glob('/content/trafficcam/**/frame0.json', recursive=True))
    assert _n >= 78, f'STOP: expected >=78 videos under /content/trafficcam, found {_n}'
    print(f'download ok: {_n} videos')
    !cd /content/SGP-IV && python scripts/prepare_dataset.py /content/trafficcam \
        --out /content/tcam --source trafficcam --option B
    !cd /content/SGP-IV && python scripts/prepare_dataset.py /content/tcam --make-calibration
    !cd /content/SGP-IV && python scripts/prepare_dataset.py /content/tcam --check
    import os as _os
    assert _os.path.exists('/content/tcam/data.yaml'), 'STOP: converter failed — read output above'
else:
    print('tcam exists, skipped')
# ITD later: convert with its own --source into /content/itd, then merge
# train/val/test image+label dirs into /content/tcam (same 6-class ids).


In [ ]:
# Cell 2 — train: yolov8n from loop-8 weights, low-LR polish (finale recipe)
# Upload best_f007.pt to /content when prompted (Files pane > Upload).
# PREREQ: Cell 1 must have produced /content/tcam/data.yaml (run it first).
import os
assert os.path.exists('/content/best_f007.pt'), 'upload best_f007.pt first'
assert os.path.exists('/content/tcam/data.yaml'), \
    'STOP: run Cell 1 (data download + convert) first — /content/tcam/data.yaml is missing'
from ultralytics import YOLO
model = YOLO('/content/best_f007.pt')
model.train(data='/content/tcam/data.yaml', epochs=10, imgsz=640, batch=32, workers=4,
              lr0=0.002, patience=10, project='/content/runs', name='tcam_loop',
              seed=42, verbose=True)
print('train done')

In [ ]:
# Cell 3 — validate: TrafficCAM val + (optional) BMD-45 no-forgetting anchor
# Anchor: mount Drive with BMD-45 val converted set, set BMD_VAL to its data.yaml.
BMD_VAL = ''  # e.g. '/content/drive/MyDrive/bmd45/data.yaml' — leave empty to skip
import glob
w = sorted(glob.glob('/content/runs/tcam_loop/weights/best.pt'))[-1]
print('weights:', w)
!yolo val model={w} data=/content/tcam/data.yaml split=val 2>&1 | tail -8
if BMD_VAL:
    !yolo val model={w} data={BMD_VAL} split=val 2>&1 | tail -8
    print('GATE: overall mAP50 must stay within -0.02 of 0.8477')

In [ ]:
# Cell 4 — export ONNX opset17 + int8 (paths derived, never aliased)
import glob, os
from ultralytics import YOLO
w = sorted(glob.glob('/content/runs/tcam_loop/weights/best.pt'))[-1]
onnx = YOLO(w).export(format='onnx', opset=17, dynamic=False)
print('fp32:', onnx)

import cv2, numpy as np, onnxruntime
from onnxruntime.quantization import (CalibrationDataReader, QuantFormat,
                                      QuantType, quantize_dynamic, quantize_static)

cals = sorted(glob.glob('/content/tcam/calibration/frames/*.jpg'))[:100]
assert cals, 'no calibration frames'
sess = onnxruntime.InferenceSession(onnx, providers=['CPUExecutionProvider'])
inp = sess.get_inputs()[0].name

class Reader(CalibrationDataReader):
    def __init__(self):
        self.it = iter(cals)
    def get_next(self):
        try:
            p = next(self.it)
        except StopIteration:
            return None
        img = cv2.resize(cv2.imread(p), (640, 640)).astype(np.float32) / 255.0
        return {inp: np.ascontiguousarray(img.transpose(2, 0, 1)[None])}

root, _ = os.path.splitext(onnx)
int8 = root + '-int8.onnx'
assert int8 != onnx, 'int8 path aliases fp32 path - refusing to overwrite fp32'

def _max_score(path):
    s = onnxruntime.InferenceSession(path, providers=['CPUExecutionProvider'])
    n = s.get_inputs()[0].name
    img = cv2.resize(cv2.imread(cals[0]), (640, 640)).astype(np.float32) / 255.0
    blob = np.ascontiguousarray(img.transpose(2, 0, 1)[None])
    out = s.run(None, {n: blob})[0]
    w_ = out[0].T if (out[0].shape[0] == 10 and out[0].shape[1] != 10) else out[0]
    return float(w_[:, 4:].max())

# Non-degeneracy gate (2026-09-21 incident: a static int8 shipped all-zero
# scores and silently killed every downstream quality number). Static first,
# dynamic fallback, fp32-only last resort - never package blind.
quant_kind = None
try:
    quantize_static(onnx, int8, Reader(), quant_format=QuantFormat.QDQ,
                    weight_type=QuantType.QInt8)
    if _max_score(int8) > 0:
        quant_kind = 'static'
        print('static int8 ok, max score:', _max_score(int8))
    else:
        print('static int8 degenerate (all-zero) - falling back to dynamic')
except Exception as e:
    print('STATIC QUANT FAILED:', e)
if quant_kind is None:
    quantize_dynamic(onnx, int8, weight_type=QuantType.QInt8)
    assert _max_score(int8) > 0, 'STOP: even dynamic quant is silent - do not package'
    quant_kind = 'dynamic'
    print('dynamic int8 ok, max score:', _max_score(int8))
print('quant:', quant_kind, '->', int8)


In [ ]:
# Cell 5 — package registry candidate + verdict
import datetime, json, os, shutil
stamp = datetime.date.today().isoformat()
reg = f'/content/registry_candidate_{stamp}'
os.makedirs(reg, exist_ok=True)
for f in [onnx, int8]:
    shutil.copy(f, reg)
shutil.copy('/content/tcam/data.yaml', os.path.join(reg, 'data.yaml'))
json.dump({'classes': ['car', 'motorcycle', 'bus', 'truck', 'bicycle', 'auto'],
             'imgsz': 640, 'source': 'trafficcam-tcam_loop', 'base': 'best_f007.pt',,
             'quant': globals().get('quant_kind', 'unknown'),
             'date': stamp}, open(os.path.join(reg, 'metadata.json'), 'w'), indent=2)
zipf = f'/content/trafficcam_candidate_{stamp}.zip'
!cd /content && zip -qr {zipf} {reg}
!ls -la {zipf}
from google.colab import files
files.download(zipf)
print('HAND-BACK:', zipf)
print('Promotion gates (applied on return): TrafficCAM mAP50 up AND BMD-Val within -0.02 of 0.8477 AND int8 fps >= 4.5 CPU. Else finale stays canonical.')